# Chapter 6: Diffusion Models — Generating Images from Noise

*Build a Multimodal Model from Scratch*

---

So far in this book we have built models that **understand** images:
ViT encodes them, CLIP aligns them with text, and the VLM generates captions.

This chapter flips the direction: we build a model that **generates** images.
Diffusion models are the engine behind Stable Diffusion, DALL·E, Imagen, and
Midjourney.  They share key components with everything we have already built —
the architecture intuition is the same, only the training objective changes.

By the end of this chapter you will understand:
1. The **forward process** — how to corrupt an image into pure noise.
2. The **reverse process** — what the neural network actually learns.
3. How to build a minimal **U-Net** conditioned on a diffusion timestep.
4. The **DDPM training and sampling** loops.
5. How to add **class-conditional generation** by injecting a label embedding.
6. The connection back to **CLIP**: how text conditioning works in Stable Diffusion.

In [ ]:
import os, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

os.makedirs('figures', exist_ok=True)
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

---
## 6.1  The Core Idea: Destroy, Then Learn to Reconstruct

The key insight behind diffusion models is a counterintuitive training trick:

> **Train a network to undo a known noise-adding process.**

Because we *control* exactly how noise is added (the forward process), we
always have ground-truth supervision.  The network never needs labels — the
noisy images are their own training signal.

### The Two Processes

```
Forward (fixed, no learning):
  x₀ ──→ x₁ ──→ x₂ ──→ ... ──→ xT
 (image)                        (pure Gaussian noise)

Reverse (learned):
  xT ──→ x_{T-1} ──→ ... ──→ x₁ ──→ x₀
(noise)                              (generated image)
```

At inference time, we sample a random Gaussian vector `xT ~ N(0, I)`
and run the reverse process T times to get a generated image `x₀`.

---
## 6.2  The Forward Process: Adding Noise Gradually

### The Math

At each step t, we add a small amount of Gaussian noise controlled by a
noise schedule β₁, β₂, ..., βT:

```
q(xₜ | xₜ₋₁) = N(xₜ ;  √(1 - βₜ) · xₜ₋₁,  βₜ · I)
```

The key property of Gaussian distributions lets us jump directly to
any timestep t without iterating through all prior steps:

```
q(xₜ | x₀) = N(xₜ ;  √ᾱₜ · x₀,  (1 - ᾱₜ) · I)
```

where  `ᾱₜ = ∏ₛ₌₁ᵗ (1 - βₛ)` (cumulative product of signal retention).

**In code:**

```python
xt = sqrt_alpha_bar[t] * x0 + sqrt_one_minus_alpha_bar[t] * noise
```

This is the "reparameterisation trick" — sample one Gaussian `noise ~ N(0, I)`
and combine it with the clean image deterministically.

### The Noise Schedule

Linear schedule: β increases linearly from β_start to β_end over T steps.
The model sees slightly more noise at each step, giving it a curriculum.

In [ ]:
class NoiseSchedule:
    """
    Linear DDPM noise schedule.

    Precomputes alpha, alpha_bar and their square roots for fast
    indexing during training and sampling.
    """

    def __init__(self, T: int = 1000,
                 beta_start: float = 1e-4,
                 beta_end:   float = 0.02,
                 device=DEVICE):
        self.T = T

        beta             = torch.linspace(beta_start, beta_end, T, device=device)
        alpha            = 1.0 - beta
        alpha_bar        = torch.cumprod(alpha, dim=0)
        alpha_bar_prev   = F.pad(alpha_bar[:-1], (1, 0), value=1.0)

        self.beta                     = beta
        self.alpha                    = alpha
        self.alpha_bar                = alpha_bar
        self.sqrt_alpha_bar           = alpha_bar.sqrt()
        self.sqrt_one_minus_alpha_bar = (1.0 - alpha_bar).sqrt()
        # Posterior variance for reverse process
        self.posterior_variance = beta * (1 - alpha_bar_prev) / (1 - alpha_bar)

    def q_sample(self, x0: torch.Tensor, t: torch.Tensor,
                 noise: torch.Tensor | None = None) -> tuple:
        """
        Sample xₜ from q(xₜ | x₀) in one step.

        x0   : (B, C, H, W) clean images, values in [-1, 1]
        t    : (B,) timestep indices
        noise: optional pre-sampled noise; if None, sampled here

        Returns (xₜ, noise)
        """
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ab  = self.sqrt_alpha_bar[t].reshape(-1, 1, 1, 1)
        sqrt_1mb = self.sqrt_one_minus_alpha_bar[t].reshape(-1, 1, 1, 1)
        xt = sqrt_ab * x0 + sqrt_1mb * noise
        return xt, noise


schedule = NoiseSchedule(T=1000, device=DEVICE)
print(f'T={schedule.T}  |  alpha_bar[0]={schedule.alpha_bar[0]:.4f}  '
      f'alpha_bar[500]={schedule.alpha_bar[500]:.4f}  '
      f'alpha_bar[999]={schedule.alpha_bar[999]:.6f}')
print('alpha_bar[999] ≈ 0  → at t=T the image is nearly pure noise')

### Visualizing the Forward Process

Let's watch a synthetic image dissolve into noise step by step:

In [ ]:
def make_demo_image():
    """Simple 32x32 image: red circle on blue background."""
    img = torch.zeros(1, 3, 32, 32)
    img[0, 2] = 0.6          # blue background
    # draw a rough circle
    cx, cy, r = 16, 16, 8
    for i in range(32):
        for j in range(32):
            if (i - cy)**2 + (j - cx)**2 < r**2:
                img[0, 0, i, j] = 0.9   # red circle
                img[0, 2, i, j] = 0.0
    return img.to(DEVICE) * 2 - 1   # scale to [-1, 1]


def plot_forward_process(schedule, steps=(0, 100, 300, 500, 700, 900, 999)):
    x0   = make_demo_image()
    fig, axes = plt.subplots(1, len(steps), figsize=(14, 2.5))
    for ax, t_val in zip(axes, steps):
        t     = torch.tensor([t_val], device=DEVICE)
        xt, _ = schedule.q_sample(x0, t)
        img   = ((xt[0].permute(1, 2, 0).cpu().clamp(-1, 1) + 1) / 2).numpy()
        ax.imshow(img)
        ax.set_title(f't = {t_val}', fontsize=9)
        ax.axis('off')
    plt.suptitle('Forward Process: Clean Image → Pure Noise',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('figures/ch06_forward.png', dpi=110, bbox_inches='tight')
    plt.show()

plot_forward_process(schedule)

---
## 6.3  What the Network Learns

Given a noisy image `xₜ` and the timestep `t`, the network predicts the
noise `ε` that was added:

```
ε_θ(xₜ, t)  ≈  ε
```

The training objective is simply mean squared error between predicted
and actual noise:

```
L = E_{x₀, t, ε} [ || ε - ε_θ(xₜ, t) ||² ]
```

This is called **ε-prediction** (epsilon prediction).  The network does NOT
predict the clean image directly — it predicts the noise direction, which
empirically gives better results.

### Why Does Predicting Noise Work?

Rearranging `xₜ = √ᾱₜ x₀ + √(1-ᾱₜ) ε`, if we know `xₜ` and predict `ε`,
we can recover `x₀`:

```
x̂₀ = (xₜ - √(1-ᾱₜ) ε_θ) / √ᾱₜ
```

So predicting noise is equivalent to predicting the clean image — just a
linear change of variables.  But the noise formulation keeps gradients
numerically stable across all timesteps.

---
## 6.4  Sinusoidal Time Embedding

The network needs to know *how noisy* the current image is.  We encode
the timestep `t` as a fixed sinusoidal vector, exactly like the positional
encoding in the original Transformer.

```
PE(t, 2i)   = sin(t / 10000^(2i / d))
PE(t, 2i+1) = cos(t / 10000^(2i / d))
```

This embedding is then projected through a small MLP so the network can
learn a flexible time-conditioning signal.

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    """
    Fixed sinusoidal timestep embedding, projected through a small MLP.

    This is the standard way to condition the noise predictor on t.
    Sinusoidal encoding has the useful property that nearby timesteps
    produce similar embeddings.
    """

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        # Project the sinusoidal encoding to a richer representation
        self.proj = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim),
        )

    def _sinusoidal(self, t: torch.Tensor) -> torch.Tensor:
        """t: (B,) integer timesteps → (B, dim) sinusoidal embedding"""
        device  = t.device
        half    = self.dim // 2
        freqs   = torch.exp(
            -math.log(10000) * torch.arange(half, device=device) / (half - 1)
        )
        args    = t[:, None].float() * freqs[None, :]   # (B, half)
        return torch.cat([args.sin(), args.cos()], dim=-1)  # (B, dim)

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.proj(self._sinusoidal(t))   # (B, dim)


# Quick test
te  = SinusoidalTimeEmbedding(dim=64).to(DEVICE)
t   = torch.tensor([0, 100, 500, 999], device=DEVICE)
out = te(t)
print(f'Time embedding: t.shape={t.shape}  out.shape={out.shape}')

---
## 6.5  The Noise Predictor: A Simple U-Net

The standard architecture for the noise predictor is a **U-Net** —
an encoder-decoder with skip connections.

```
Input xₜ                        Output ε̂
(B, C, H, W)                   (B, C, H, W)

Encoder:  32×32 → 16×16 → 8×8
Bottleneck:       8×8
Decoder:   8×8  → 16×16 → 32×32
Skip connections ─────────────┘
```

At each layer, the **time embedding** is added as a bias after a linear
projection, conditioning every feature map on the current timestep.

For conditional generation, a **class embedding** is also added here.

### Why U-Net?

The skip connections carry fine-grained spatial detail from encoder to
decoder.  Without them, the decoder loses the precise location of edges
and textures that make a generated image look sharp.

In [ ]:
class ResBlock(nn.Module):
    """
    Residual block with timestep (and optional class) conditioning.

    Architecture:
        x → Norm → SiLU → Conv → + time_emb → Norm → SiLU → Conv → + residual
    """

    def __init__(self, in_ch: int, out_ch: int, time_dim: int, num_classes: int = 0):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)

        # Time conditioning: project time_emb to out_ch and add as bias
        self.time_proj = nn.Linear(time_dim, out_ch)

        # Class conditioning (optional)
        self.class_proj = nn.Embedding(num_classes, out_ch) if num_classes > 0 else None

        # Residual projection if channels change
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor,
                class_id: torch.Tensor | None = None) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))

        # Add time embedding (broadcast over spatial dims)
        h = h + self.time_proj(F.silu(t_emb))[:, :, None, None]

        # Add class embedding if provided
        if class_id is not None and self.class_proj is not None:
            h = h + self.class_proj(class_id)[:, :, None, None]

        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.res_conv(x)   # residual connection


class SimpleUNet(nn.Module):
    """
    A minimal but complete U-Net for 32×32 images.

    3 resolution levels: 32→16→8 (encoder), 8→16→32 (decoder).
    Time embedding injected at every residual block.
    Optional class embedding for conditional generation.

    Args:
        in_channels  : image channels (3 for RGB)
        base_ch      : base channel count (doubles at each level)
        time_dim     : sinusoidal time embedding dimension
        num_classes  : number of classes for conditional generation (0 = unconditional)
    """

    def __init__(self, in_channels=3, base_ch=64, time_dim=128, num_classes=0):
        super().__init__()
        C = base_ch

        # Time embedding
        self.time_embed = SinusoidalTimeEmbedding(time_dim)

        # ── Encoder ─────────────────────────────────────────────────────────
        self.enc_in    = nn.Conv2d(in_channels, C, 3, padding=1)
        self.enc1      = ResBlock(C,   C,   time_dim, num_classes)   # 32×32
        self.down1     = nn.Conv2d(C,   C*2, 3, stride=2, padding=1) # 16×16
        self.enc2      = ResBlock(C*2, C*2, time_dim, num_classes)   # 16×16
        self.down2     = nn.Conv2d(C*2, C*4, 3, stride=2, padding=1) # 8×8

        # ── Bottleneck ───────────────────────────────────────────────────────
        self.mid1 = ResBlock(C*4, C*4, time_dim, num_classes)
        self.mid2 = ResBlock(C*4, C*4, time_dim, num_classes)

        # ── Decoder ─────────────────────────────────────────────────────────
        self.up2      = nn.ConvTranspose2d(C*4, C*2, 2, stride=2)    # 16×16
        self.dec2     = ResBlock(C*4, C*2, time_dim, num_classes)     # concat enc2 skip
        self.up1      = nn.ConvTranspose2d(C*2, C, 2, stride=2)      # 32×32
        self.dec1     = ResBlock(C*2, C, time_dim, num_classes)       # concat enc1 skip

        # ── Output ──────────────────────────────────────────────────────────
        self.out_norm = nn.GroupNorm(8, C)
        self.out_conv = nn.Conv2d(C, in_channels, 1)

    def forward(self, x: torch.Tensor, t: torch.Tensor,
                class_id: torch.Tensor | None = None) -> torch.Tensor:
        t_emb = self.time_embed(t)                 # (B, time_dim)

        # Encoder
        x1 = self.enc1(self.enc_in(x), t_emb, class_id)   # 32×32
        x2 = self.enc2(self.down1(x1), t_emb, class_id)   # 16×16
        x3 = self.down2(x2)                                # 8×8

        # Bottleneck
        x3 = self.mid1(x3, t_emb, class_id)
        x3 = self.mid2(x3, t_emb, class_id)

        # Decoder with skip connections
        h = self.dec2(torch.cat([self.up2(x3), x2], dim=1), t_emb, class_id)
        h = self.dec1(torch.cat([self.up1(h),  x1], dim=1), t_emb, class_id)

        return self.out_conv(F.silu(self.out_norm(h)))


# Sanity check
model = SimpleUNet(in_channels=3, base_ch=32, time_dim=64, num_classes=4).to(DEVICE)
x     = torch.randn(2, 3, 32, 32, device=DEVICE)
t     = torch.randint(0, 1000, (2,), device=DEVICE)
cid   = torch.tensor([0, 1], device=DEVICE)
out   = model(x, t, cid)
print(f'U-Net I/O: {x.shape} → {out.shape}  (same shape)')
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total:,}')

---
## 6.6  Synthetic Dataset: Geometric Shapes

We generate 4 classes of 32×32 images with distinct visual structure.
The model needs to learn to generate each class — simple enough to converge
in minutes on a CPU.

| Class | Description |
|-------|-------------|
| 0 | Red circle on dark background |
| 1 | Blue circle on dark background |
| 2 | Red square on dark background |
| 3 | Blue square on dark background |

In [ ]:
class ShapeDataset(Dataset):
    """4-class geometric shapes, values in [-1, 1]."""

    def __init__(self, n_per_class=500, img_size=32, seed=42):
        rng = np.random.default_rng(seed)
        imgs, labels = [], []
        for label in range(4):
            color_ch = 0 if label in (0, 2) else 2   # 0=red, 2=blue
            is_circle = label in (0, 1)
            for _ in range(n_per_class):
                img = np.full((3, img_size, img_size), -0.8, dtype=np.float32)
                cx = rng.integers(10, 22); cy = rng.integers(10, 22)
                r  = rng.integers(6, 10)
                for i in range(img_size):
                    for j in range(img_size):
                        inside = ((i-cy)**2 + (j-cx)**2 < r**2
                                  if is_circle else
                                  abs(i-cy) < r and abs(j-cx) < r)
                        if inside:
                            img[color_ch, i, j] = 0.9
                img += rng.normal(0, 0.05, img.shape).astype(np.float32)
                imgs.append(torch.tensor(img.clip(-1, 1)))
                labels.append(label)
        self.imgs   = imgs
        self.labels = labels

    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.imgs[i], self.labels[i]


dataset = ShapeDataset(n_per_class=500)
loader  = DataLoader(dataset, batch_size=64, shuffle=True)
print(f'Dataset: {len(dataset)} images, 4 classes')

# Visualise one sample per class
fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
shown = set()
for img, lbl in dataset:
    if lbl not in shown:
        shown.add(lbl)
        ax = axes[lbl]
        ax.imshow(((img.permute(1,2,0) + 1) / 2).clamp(0,1).numpy())
        ax.set_title(['Red circle','Blue circle','Red square','Blue square'][lbl], fontsize=9)
        ax.axis('off')
    if len(shown) == 4: break
plt.suptitle('Training Samples (one per class)', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/ch06_samples.png', dpi=110, bbox_inches='tight')
plt.show()

---
## 6.7  Training Loop

The training loop is elegantly simple:

```
for each batch x₀:
    1. Sample random timesteps t ~ Uniform(0, T)
    2. Sample noise  ε ~ N(0, I)
    3. Compute noisy image  xₜ = √ᾱₜ x₀ + √(1-ᾱₜ) ε
    4. Predict noise  ε̂ = ε_θ(xₜ, t, class_label)
    5. Loss = MSE(ε, ε̂)
    6. Backpropagate
```

Notice there are **no discriminators, no adversarial training** — just
straightforward regression.  This is why diffusion models train so stably
compared to GANs.

In [ ]:
model     = SimpleUNet(in_channels=3, base_ch=32, time_dim=64, num_classes=4).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
scheduler_lr = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80)

EPOCHS = 80
losses = []

print(f'Training DDPM for {EPOCHS} epochs ...')
for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for x0, class_ids in loader:
        x0        = x0.to(DEVICE)
        class_ids = class_ids.to(DEVICE)
        B         = x0.shape[0]

        # Step 1-2: random t and noise
        t     = torch.randint(0, schedule.T, (B,), device=DEVICE)
        noise = torch.randn_like(x0)

        # Step 3: corrupt the image
        xt, _ = schedule.q_sample(x0, t, noise)

        # Step 4: predict noise
        pred_noise = model(xt, t, class_ids)

        # Step 5: MSE loss
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    scheduler_lr.step()
    avg = epoch_loss / len(loader)
    losses.append(avg)
    if epoch % 10 == 0:
        print(f'  Epoch {epoch:3d}/{EPOCHS}  loss={avg:.5f}')

print('\nTraining complete.')

# Plot loss curve
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(losses, color='#3b82f6', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('DDPM Training Loss', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/ch06_loss.png', dpi=110, bbox_inches='tight')
plt.show()

---
## 6.8  DDPM Sampling: The Reverse Process

To generate an image, we start from `xT ~ N(0, I)` and iteratively
denoise it using the learned model.

At each step t, we compute the posterior mean using the predicted noise,
then add a small amount of noise (except at t=0):

```
μ_θ(xₜ, t) = (1/√αₜ) * (xₜ - (βₜ/√(1-ᾱₜ)) * ε_θ(xₜ, t))

x_{t-1} = μ_θ + √(posterior_variance) * z    (z ~ N(0,I) if t > 0, else 0)
```

This iterative denoising is what makes diffusion models slow at inference —
we must run the network T times.  DDIM and other samplers reduce this to
10–50 steps, but here we use the original DDPM 1000-step schedule.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, schedule, shape, class_id: int,
                device=DEVICE, n_steps: int | None = None) -> torch.Tensor:
    """
    Generate one image via DDPM reverse process.

    model    : trained SimpleUNet
    schedule : NoiseSchedule
    shape    : (C, H, W)
    class_id : class label for conditional generation
    n_steps  : if < T, use a strided subset of timesteps (fast approximate sampling)
    """
    model.eval()
    T_total = schedule.T
    steps   = list(range(T_total - 1, -1, -1))
    if n_steps is not None and n_steps < T_total:
        # Uniformly subsample timesteps
        stride = T_total // n_steps
        steps  = list(range(T_total - 1, -1, -stride))

    # Start from pure noise
    x = torch.randn(1, *shape, device=device)
    cid = torch.tensor([class_id], device=device)

    for t_val in steps:
        t     = torch.tensor([t_val], device=device)
        alpha = schedule.alpha[t_val]
        ab    = schedule.alpha_bar[t_val]
        ab_sq = schedule.sqrt_alpha_bar[t_val]
        smab  = schedule.sqrt_one_minus_alpha_bar[t_val]
        beta  = schedule.beta[t_val]

        # Predict noise
        pred_noise = model(x, t, cid)

        # Compute predicted x₀
        x0_pred = (x - smab * pred_noise) / ab_sq

        # Posterior mean
        coef1 = beta * ab_sq / smab
        coef2 = (1 - ab) * schedule.sqrt_alpha_bar[t_val-1] / smab if t_val > 0 else 0
        mean  = (x - coef1 * pred_noise) / alpha.sqrt()

        if t_val > 0:
            var   = schedule.posterior_variance[t_val].clamp(min=1e-20)
            noise = torch.randn_like(x)
            x     = mean + var.sqrt() * noise
        else:
            x = mean

    return x.clamp(-1, 1)


# Generate one sample per class
print('Generating samples (this takes a moment with 1000 steps) ...')
fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
class_names = ['Red circle', 'Blue circle', 'Red square', 'Blue square']
for c in range(4):
    sample = ddpm_sample(model, schedule, (3, 32, 32), class_id=c)
    img    = ((sample[0].permute(1,2,0).cpu() + 1) / 2).clamp(0,1).numpy()
    axes[c].imshow(img)
    axes[c].set_title(class_names[c], fontsize=9)
    axes[c].axis('off')
plt.suptitle('Generated Samples (class-conditional DDPM)', fontweight='bold')
plt.tight_layout()
plt.savefig('figures/ch06_generated.png', dpi=110, bbox_inches='tight')
plt.show()

---
## 6.9  Visualizing the Denoising Trajectory

The most compelling demonstration of a diffusion model is watching it
gradually reveal an image from noise.

In [ ]:
@torch.no_grad()
def sample_with_trajectory(model, schedule, shape, class_id: int,
                            save_steps=(999, 800, 600, 400, 200, 100, 50, 0),
                            device=DEVICE):
    """Run full DDPM reverse process, saving xₜ at specified timesteps."""
    model.eval()
    x   = torch.randn(1, *shape, device=device)
    cid = torch.tensor([class_id], device=device)
    snapshots = {}

    for t_val in range(schedule.T - 1, -1, -1):
        t     = torch.tensor([t_val], device=device)
        alpha = schedule.alpha[t_val]
        smab  = schedule.sqrt_one_minus_alpha_bar[t_val]
        ab_sq = schedule.sqrt_alpha_bar[t_val]
        beta  = schedule.beta[t_val]

        pred_noise = model(x, t, cid)
        mean = (x - beta / smab * pred_noise) / alpha.sqrt()

        if t_val > 0:
            var  = schedule.posterior_variance[t_val].clamp(min=1e-20)
            x    = mean + var.sqrt() * torch.randn_like(x)
        else:
            x = mean

        if t_val in save_steps:
            snapshots[t_val] = x.clamp(-1, 1).clone()

    return snapshots


print('Recording denoising trajectory ...')
traj = sample_with_trajectory(model, schedule, (3, 32, 32), class_id=0)

steps_sorted = sorted(traj.keys(), reverse=True)
fig, axes = plt.subplots(1, len(steps_sorted), figsize=(14, 2.5))
for ax, t_val in zip(axes, steps_sorted):
    img = ((traj[t_val][0].permute(1,2,0).cpu() + 1) / 2).clamp(0,1).numpy()
    ax.imshow(img)
    ax.set_title(f't={t_val}', fontsize=9)
    ax.axis('off')
plt.suptitle('Denoising Trajectory: Noise → Image (Red Circle)',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('figures/ch06_trajectory.png', dpi=110, bbox_inches='tight')
plt.show()

---
## 6.10  Text Conditioning: The CLIP Connection

In our simple demo above, we conditioned generation on an integer class label.
In production systems like Stable Diffusion, the condition is a **text
embedding from CLIP**.

The connection to Chapter 2 is direct:

```
User prompt: "a red circle"
         ↓  CLIP Text Encoder
Text embedding: (1, proj_dim)
         ↓  injected into each ResBlock of the U-Net
Generated image matching the description
```

The U-Net's `ResBlock` receives the text embedding instead of (or in addition to)
a class embedding:

```python
# Instead of:
h = h + self.class_proj(class_id)[:, :, None, None]

# Stable Diffusion does:
h = h + self.text_proj(text_emb)[:, :, None, None]
# or via cross-attention between h and text_emb
```

### Classifier-Free Guidance (CFG)

Real text-to-image models use **classifier-free guidance** for sharper,
more text-aligned outputs:

1. Train the model with text conditioning dropped at random (10-20% of batches
   replaced with a null embedding `∅`).
2. At inference, run the U-Net **twice** per step:
   - Once conditioned on the text embedding `c`.
   - Once with the null embedding `∅`.
3. Combine predictions with a guidance scale `w`:

```
ε_guided = ε_∅ + w · (ε_c - ε_∅)
```

Higher `w` (e.g., 7.5) → more faithful to the text but less diverse.
Lower `w` → more diverse but may ignore the text.

This is the `guidance_scale` parameter you see in Stable Diffusion's API.

---
## 6.11  Latent Diffusion: Why Stable Diffusion Is Fast

Our model runs diffusion directly on 32×32 pixel images.
Real models like Stable Diffusion work in **latent space** instead:

```
Image (512×512×3)
    ↓  VAE Encoder  (pretrained, frozen)
Latent (64×64×4)    ← 48× fewer elements!
    ↓  Diffusion U-Net  (1000 steps of denoising)
Latent (64×64×4)
    ↓  VAE Decoder  (pretrained, frozen)
Image (512×512×3)
```

Key advantages:
1. **Speed** — the U-Net operates on 64×64 instead of 512×512: 64× fewer pixels.
2. **Quality** — the VAE has already learned a meaningful compact representation;
   the U-Net only needs to model structure, not low-level pixel statistics.
3. **Modularity** — the VAE and U-Net can be trained separately.

The VAE here is an **Autoencoder** with a KL-regularized bottleneck, trained
on image reconstruction.  It is separate from the CLIP model, though both
share the goal of compressing images to a compact representation.

| Model | Input | Output | Role |
|-------|-------|--------|------|
| CLIP Image Encoder | 224×224 image | 512-dim vector | Semantic similarity |
| VAE Encoder | 512×512 image | 64×64×4 tensor | Spatial compression |
| Diffusion U-Net | 64×64×4 noisy latent | 64×64×4 noise pred | Generation |
| VAE Decoder | 64×64×4 latent | 512×512 image | Pixel reconstruction |

---
## 6.12  Chapter Summary

| Concept | Key Idea |
|---------|----------|
| **Forward process** | Add Gaussian noise over T steps; closed-form: `xₜ = √ᾱₜ x₀ + √(1-ᾱₜ) ε` |
| **Reverse process** | Learned denoising: `x_{t-1} = f(xₜ, t, ε_θ)` |
| **ε-prediction** | Network predicts the added noise, not the clean image; numerically stable |
| **Noise schedule** | βₜ controls how much noise at each step; linear or cosine |
| **Time embedding** | Sinusoidal encoding + MLP tells the U-Net how noisy the image is |
| **U-Net** | Encoder-decoder with skip connections; preserves spatial detail |
| **Class conditioning** | Inject class embedding into each ResBlock for guided generation |
| **CFG** | Run conditioned + unconditioned; interpolate for text fidelity vs. diversity |
| **Latent diffusion** | Diffuse in VAE latent space; 48× faster than pixel-space diffusion |

---

## Book Complete

You have now built the full multimodal stack from scratch:

| Chapter | Model | Direction |
|---------|-------|-----------|
| Ch 1 | Vision Transformer | Image → features |
| Ch 2 | CLIP | Image + Text → shared embedding |
| Ch 3–4 | VLM (LLaVA-style) | Image → Text (captioning / VQA) |
| Ch 5 | Inference | Text generation strategies |
| **Ch 6** | **DDPM / Stable Diffusion** | **Text → Image (generation)** |

Together these two directions — understanding (Ch 1–5) and generation (Ch 6) —
form the complete picture of modern multimodal AI.